In [1]:
#Imports
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


import pandas as pd
import numpy as np
from scipy.linalg import block_diag

from model_funcs import run_em, sort_matrix, create_Y

# Bayesian Mixture Model

**Goal**: Model the uncertainty expressed by multiple annotations
**Tool**: Multinomial Mixture Model
- There exists a latent ground truth, i.e. a "true label" for each image
- This latent ground truth is unambiguous but the annotator’s opinion about it is not
- We set up a distributional framework and model the observed annotations under the assumption that there exists a latent true label
- The parameters of the associated distributions allow us to analyse the sources of labelling uncertainty and we can estimate the latent labels through the posterior probabilities

Assume latent ground truth:

$Z^{(i)}\sim Multi(\boldsymbol{\pi}, 1)$ iid.  $\rightarrow$ prior

Given latent true class, the labellers' votes follow:

$Y^{(i)}|Z^{(i)}\sim Multi(\boldsymbol{\theta_{Z^{(i)}}}, J^i)$ iid. $\rightarrow$ likelihood

or explicitly:

$Y^{(i)}|(Z^{(i)} = airplane) \sim Multi(\boldsymbol{\theta_{airplane}}, J^i) \\
Y^{(i)}|(Z^{(i)} = automobile) \sim Multi(\boldsymbol{\theta_{automobile}}, J^i)\\
Y^{(i)}|(Z^{(i)} = bird) \sim Multi(\boldsymbol{\theta_{bird}}, J^i)\\
Y^{(i)}|(Z^{(i)} = cat) \sim Multi(\boldsymbol{\theta_{cat}}, J^i)\\
Y^{(i)}|(Z^{(i)} = deer) \sim Multi(\boldsymbol{\theta_{deer}}, J^i)\\
Y^{(i)}|(Z^{(i)} = dog) \sim Multi(\boldsymbol{\theta_{dog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = frog) \sim Multi(\boldsymbol{\theta_{frog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = horse) \sim Multi(\boldsymbol{\theta_{horse}}, J^i)\\
Y^{(i)}|(Z^{(i)} = ship) \sim Multi(\boldsymbol{\theta_{ship}}, J^i)\\
Y^{(i)}|(Z^{(i)} = truck) \sim Multi(\boldsymbol{\theta_{truck}}, J^i).$


Use Bayes Rule to model latent ground truth given votes:

$\tau^{(i)}_l = P(Z^{(i)}=l|Y^{(i)}) = \frac{P(Z^{(i)}=l) \cdot P(Y^{(i)}|Z^{(i)} = l)}{P(Y^{(i)})} = \frac{prior \cdot likelihood}{evidence}$



Apply Expectation Maximization (EM) algorithm, iteratively estimate latent ground truth and parameters

1. E-Step:
calculate posterior class probabilities $\tau^{(i)}_l$ given $\pi$ and $\theta_{Z^{(i)}}$, i.e., parameters of prior and likelihood
choose class with highest posterior as latent ground truth class $Z^{(i)}$

2. M-Step:
update $\pi$ and $\theta_{Z^{(i)}}$ given $\tau^{(i)}_l$, i.e.,  posterior class probabilities

Initial values:
- $\pi$ = (1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10)
- $\theta$ drawn from dirichlet with $\alpha$ = (20,20,20,20,20,20,20,20,20,20) (i.e., 2* number_of_observed_classes)

# 1. Run SEM

In [13]:
# load data
cifar10h = pd.read_csv('../data/cifar10h_S.csv', index_col=0)
cifar10h_entropy = pd.read_csv('../data/cifar10h_Se.csv', index_col=0)

# convert the data to the vote distribution matrix Y
Y = create_Y(cifar10h)
Y_entropy = create_Y(cifar10h_entropy)

#extract relevant columns and convert to one-hot-encoded numpy array
Y_one_hot = np.array(Y[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]).astype(int)
Y_entropy_one_hot = np.array(Y_entropy[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]).astype(int)

Next we run the SEM algorithm on the CIFAR-10 dataset, denoted by $\bm{Y}$. The algorithms returns:

0. $\ell(\bm{Y}; \hat{\bm{\pi}}, \hat{\Theta})$
1. $\hat{\bm{\pi}}$
2. $\hat{\Theta}$
3. $\hat{\tau}$
4. $(\hat{\Theta}_t)_{t=1,\dots,T}$
5. $(\hat{\bm{\pi}}_t)_{t=1,\dots,T}$
6. $(\hat{\tau}_t^{(i)})_{t=1,\dots,T}$
7. $(Z_t^{(i)})_{t=1,\dots,T}$
8. $\hat{\bm{\vartheta}}$
9. $\widehat{\mathbb{V}}(\hat{\bm{\vartheta}})$

In [15]:
# run sem on cifar10h
cifar10h_sem = run_em(Y_one_hot, K=10, max_iter=150)
cifar10h_sem_entropy = run_em(Y_entropy_one_hot, K=10, max_iter=150)

In [16]:
# sort the confusion matrix
confusion_matrix = sort_matrix(cifar10h_sem[2])
confusion_matrix_entropy = sort_matrix(cifar10h_sem_entropy[2])

In [17]:
# Convert theta_matched, pi_matched, and tau_matched to DataFrames
theta = pd.DataFrame(confusion_matrix)
pi = pd.DataFrame(cifar10h_sem[1])
tau = pd.DataFrame(cifar10h_sem[3])

theta_entropy = pd.DataFrame(confusion_matrix_entropy)
pi_entropy = pd.DataFrame(cifar10h_sem_entropy[1])
tau_entropy = pd.DataFrame(cifar10h_sem_entropy[3])

We compute the posterior entropy of the images using the posterior probabilities $\hat{\tau}$.
We define 
$$H^{(i)} = -\sum_{z=1}^{10} \hat{\tau}_z^{(i)} \log_2(\hat{\tau}_z^{(i)})$$

In [18]:
# compute the posterior entropy
post_entropy = pd.DataFrame(-np.sum(cifar10h_sem[3] * np.log2(cifar10h_sem[3]), axis=1))
post_entropy_entropy = pd.DataFrame(-np.sum(cifar10h_sem_entropy[3] * np.log2(cifar10h_sem_entropy[3]), axis=1))

Next, we will compute the varince of the estimated parameter $\hat{\Theta}$ and the vectorized version of the estimated parameter $\hat{\bm{\vartheta}}$.


In [19]:
theta_old = pd.DataFrame(cifar10h_sem[8])
variance_theta = pd.DataFrame(cifar10h_sem[9])

theta_entropy_old = pd.DataFrame(cifar10h_sem_entropy[8])
variance_theta_entropy = pd.DataFrame(cifar10h_sem_entropy[9])

In [20]:
# export the results to csv
theta.to_csv('../data/theta_S.csv', index=False)
pi.to_csv('../data/pi_S.csv', index=False)
tau.to_csv('../data/tau_S.csv', index=False)
post_entropy.to_csv('../data/post_entropy_S.csv', index=False)
theta_old.to_csv('../data/theta_old_S.csv', index=False)
variance_theta.to_csv('../data/variance_theta_S.csv', index=False)

theta_entropy.to_csv('../data/theta_Se.csv', index=False)
pi_entropy.to_csv('../data/pi_Se.csv', index=False)
tau_entropy.to_csv('../data/tau_Se.csv', index=False)
post_entropy_entropy.to_csv('../data/post_entropy_Se.csv', index=False)
theta_entropy_old.to_csv('../data/theta_old_Se.csv', index=False)
variance_theta_entropy.to_csv('../data/variance_theta_Se.csv', index=False)
